# Spark SQL Fundamentals

*This notebook focuses on how Spark represents and queries structured data using Spark SQL.*
*It assumes familiarity with SparkSession and basic Spark execution concepts.*

---


## What is Spark SQL?

Spark SQL is a **module of Apache Spark** that allows working with structured data using SQL-like queries.

It is important to clarify that Spark SQL:

- Is not a database
- It does not store data
- It does not manage persistence by itself

Instead, Spark SQL is a **query and execution engine**.

It provides:

- A SQL interface (spark.sql(...))
- A way to treat DataFrames as tables
- An optimizer that decides how queries are executed in a distributed way


---

## DataFrames and Spark SQL

Spark SQL does not query files or databases directly.
It queries **DataFrames**.

Once data is represented as a DataFrame:
- it can be queried using SQL
- it can be transformed using the DataFrame API
- both approaches produce the same execution plan

This makes DataFrames the *bridge* between data sources and SQL queries.

---

## Working with SQL and DataFrames

To start working with with SQL, we need to initialize our session and load some data.

In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession
         .builder
         .appName("SparkSQL")
         .master("local[*]")
         .getOrCreate())

filePath = "data/people.json"
df = spark.read.json(filePath)

df.show(10)
df.printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/09 15:30:20 WARN Utils: Your hostname, MacBook-Air-de-Martin.local, resolves to a loopback address: 127.0.0.1; using 192.168.4.46 instead (on interface en0)
26/02/09 15:30:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/09 15:30:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/09 15:30:21 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


+---------------+---------------+-------------------+--------------------+-----------------+--------------------+--------------------+
|_corrupt_record|           city|         creditcard|               email|              mac|                name|           timestamp|
+---------------+---------------+-------------------+--------------------+-----------------+--------------------+--------------------+
|              [|           NULL|               NULL|                NULL|             NULL|                NULL|                NULL|
|           NULL|Lake Gladysberg|1228-1221-1221-1431|katlyn@jenkinsmag...|08:fd:0b:cd:77:f7|        Keeley Bosco|2015-04-25 13:57:...|
|           NULL|           NULL|1228-1221-1221-1431|juvenal@johnston....|90:4d:fa:42:63:a2|         Rubye Jerde|2015-04-25 09:02:...|
|           NULL|           NULL|               NULL|                NULL|f9:0e:d3:40:cb:e9|Miss Darian Breit...|2015-04-25 13:16:...|
|           NULL|           NULL|1228-1221-1221-1431|em

In this example we load our dataset from a JSON file, however, we can use any other source, or even load it explicitly.

This DataFrame:
- exists only in Spark
- has a schema

Now, in order to use SQL to query it, we must register this DataFrame as a **temporary view**:


In [2]:
df.createOrReplaceTempView("People")

- This does not copy data.
- It simply registers a logical name (`People`) in the SparkSession catalog.

The view:
- "lives" only for this SparkSession
- disappears when the session stops
- points to the DataFrame’s logical plan

Now we can query the DataFrame using SQL syntax:

In [3]:
spark.sql("""
          SELECT name, email
          FROM People
          WHERE name IS NOT NULL
          """).show(10)

# The SQL query must be passed as a string. In this case we use """ to start a string block,
# making the sintax more readable. It could also be passed in a single line with single "s:

# spark.sql("SELECT name, email FROM People WHERE name IS NOT NULL").show(10)

+--------------------+--------------------+
|                name|               email|
+--------------------+--------------------+
|        Keeley Bosco|katlyn@jenkinsmag...|
|         Rubye Jerde|juvenal@johnston....|
|Miss Darian Breit...|                NULL|
|    Celine Ankunding|emery_kunze@rogah...|
|    Dr. Araceli Lang|mavis_lehner@jaco...|
|         Esteban Von|                NULL|
|      Everette Swift|gielle_jacobs@fla...|
|       Terrell Boyle|augustine.conroy@...|
|   Miss Emmie Muller|                NULL|
|        Libby Renner|                NULL|
+--------------------+--------------------+
only showing top 10 rows


Important points:
- `spark.sql(...)` queries the catalog
- `People` refers to the temporary view we created
- execution happens only when `.show()` is called

The exact same logic can be expressed without SQL, by using the DataFrame API:

In [4]:
df.filter(df["name"].isNotNull())\
  .select("name", "email")\
  .show(10)

+--------------------+--------------------+
|                name|               email|
+--------------------+--------------------+
|        Keeley Bosco|katlyn@jenkinsmag...|
|         Rubye Jerde|juvenal@johnston....|
|Miss Darian Breit...|                NULL|
|    Celine Ankunding|emery_kunze@rogah...|
|    Dr. Araceli Lang|mavis_lehner@jaco...|
|         Esteban Von|                NULL|
|      Everette Swift|gielle_jacobs@fla...|
|       Terrell Boyle|augustine.conroy@...|
|   Miss Emmie Muller|                NULL|
|        Libby Renner|                NULL|
+--------------------+--------------------+
only showing top 10 rows


Even though the syntax is different:
- Spark builds the same logical plan
- the same optimizations are applied
- performance is equivalent

---

### SQL vs DataFrame API: which is better?

There is no universally “better” option.

Spark SQL is often preferable when:
- expressing complex aggregations
- working with SQL-heavy teams that are more familiar with its syntax
- readability matters

The DataFrame API is often preferable when:
- logic is dynamic or programmatic
- conditions are built in code
- integrating with Python workflows

In practice, both are often mixed in the same application.

---

## Defining a Schema 

When reading our data from an external file, Spark will infer the schema and assign types to our data, however, we might want to define our own schema because:

- Schema inference requires scanning data
- Explicit schemas skip that step
- Avoids wrong types, like assigning StingType to a numeric field.
- It is safer for production pipelines
- Aggregations and SQL queries behave correctly

Additionally, when dealing with different types of source files, Spark usually expects a certain format. In our case, the file presents a `_corrupt_record` column. This is because the source file is written as a JSON array, and Spark interprets the [] as extra unknown data. Spark expects a single object per line.

We can, on top of defining our schema, make sure Spark can read in the correct format, instead of modifying our original file:

In [5]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

# Creating the schema with our desired types, for this one, only the timestamp field was changed
myschema = StructType([
    StructField("name", StringType(), True), # the name here must match excatly with the original file
    StructField("email", StringType(), True),
    StructField("city", StringType(), True),
    StructField("mac", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("creditcard", StringType(), True)
])

# creating the dataframe, specifying the correct JSON fromat
df_2 = spark.read\
            .option("multiline", "true")\
            .schema(myschema)\
            .json(filePath)

df_2.printSchema()
df_2.show()


root
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- mac: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- creditcard: string (nullable = true)

+--------------------+--------------------+--------------------+-----------------+-------------------+-------------------+
|                name|               email|                city|              mac|          timestamp|         creditcard|
+--------------------+--------------------+--------------------+-----------------+-------------------+-------------------+
|        Keeley Bosco|katlyn@jenkinsmag...|     Lake Gladysberg|08:fd:0b:cd:77:f7|2015-04-25 08:57:36|1228-1221-1221-1431|
|         Rubye Jerde|juvenal@johnston....|                NULL|90:4d:fa:42:63:a2|2015-04-25 04:02:04|1228-1221-1221-1431|
|Miss Darian Breit...|                NULL|                NULL|f9:0e:d3:40:cb:e9|2015-04-25 08:16:03|               NULL|
|    Celine Ankunding|emer

Now, we have a cleaner view of our dataset and queries that would not have worked will work now.

We can also check the different tables we have registered in our SparkSession:

In [6]:
spark.catalog.listTables()

[Table(name='people', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

Now we will create a new Table for our cleaner version of the data, and later we will compare some queries.

In [8]:
df_2.createOrReplaceTempView("People2")
spark.catalog.listTables()

[Table(name='people', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='people2', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [20]:
# We will try to group by timestamp for the first table,
# where the field type was STRING. This wont give an error,
# but it wont access any data.
spark.sql("""
          SELECT name, email, timestamp
          FROM People
          WHERE timestamp BETWEEN
                '2015-04-25 7:00:00'
          AND   '2015-04-25 7:01:00'
          """).show()

+----+-----+---------+
|name|email|timestamp|
+----+-----+---------+
+----+-----+---------+



In [19]:
# Here, the timestamp column has the correct type,
# so we can filter by time.
spark.sql("""
          SELECT name, email, timestamp
          FROM People2
          WHERE timestamp BETWEEN
                '2015-04-25 7:00:00'
          AND   '2015-04-25 7:01:00'
          """).show()

+--------------------+--------------------+-------------------+
|                name|               email|          timestamp|
+--------------------+--------------------+-------------------+
|   Alayna Blanda Jr.|                NULL|2015-04-25 07:00:38|
|Mrs. Alberta Hackett|annette.feeney@mu...|2015-04-25 07:00:52|
|       Pamela Herman|                NULL|2015-04-25 07:00:57|
|     Gilberto Ernser|kenya_hauck@mayer...|2015-04-25 07:00:30|
|       Mr. Emmy Mann|                NULL|2015-04-25 07:00:01|
|Miss Aimee Rosenbaum|     eulah@rohan.biz|2015-04-25 07:00:59|
|      Danielle Mills|danielle.casper@m...|2015-04-25 07:00:43|
+--------------------+--------------------+-------------------+



In [21]:
spark.stop()